# Lab 04 — A calculadora de custo do DW cloud (Python, navegador)

**Onde roda:** 🟢 Browser (JupyterLite).

Objetivo: sentir **no bolso** o efeito de ler menos colunas e menos partições — o modelo
de custo *por bytes varridos* (on-demand do BigQuery).

In [ ]:
# Modelo de uma tabela: bytes de cada coluna POR PARTIÇÃO, e nº de partições.
COLUNAS = {           # bytes por partição, por coluna
    'data_pedido': 8_000_000,
    'categoria':  12_000_000,
    'estado':      6_000_000,
    'price':       8_000_000,
    'descricao':  90_000_000,   # coluna 'gorda' de texto
}
N_PARTICOES = 60      # ex.: 5 anos por mês
PRECO_POR_TB = 6.25   # US$/TB varrido (on-demand, ilustrativo)

def bytes_varridos(colunas_lidas, particoes_lidas):
    por_particao = sum(COLUNAS[c] for c in colunas_lidas)
    return por_particao * particoes_lidas

def custo_usd(bytes_):
    return bytes_ / 1e12 * PRECO_POR_TB

def relatorio(nome, colunas, particoes):
    b = bytes_varridos(colunas, particoes)
    print(f'{nome:38} {b/1e9:8.2f} GB   US$ {custo_usd(b):.4f}')

print('total da tabela =', bytes_varridos(list(COLUNAS), N_PARTICOES)/1e9, 'GB')

## Comparando estratégias de consulta
Mesma pergunta ('receita de 1 mês por categoria'), três formas — mesmo resultado, custos MUITO diferentes.

In [ ]:
# 1) SELECT * sem filtro: todas as colunas, todas as partições
relatorio('SELECT * (tudo)', list(COLUNAS), N_PARTICOES)
# 2) só as colunas necessárias, mas sem filtro de partição
relatorio('SELECT categoria,price (todas part.)', ['categoria','price'], N_PARTICOES)
# 3) colunas necessárias + filtro de 1 partição (pruning)
relatorio('SELECT categoria,price WHERE mes=X', ['categoria','price'], 1)

## Sua vez (mini-desafio)
Quantos **bytes** uma query que lê as colunas `['categoria','estado','price']` de **3**
partições varre? Calcule com `bytes_varridos(...)` e verifique.

In [ ]:
resposta = bytes_varridos(['categoria','estado','price'], 3)
resposta

In [ ]:
def verificar(v):
    esperado = (12_000_000 + 6_000_000 + 8_000_000) * 3   # = 78.000.000
    try:
        assert v == esperado, f'esperado {esperado}, veio {v}'
        print('✅ Correto! 78 MB varridos — e a query custaria US$', round(custo_usd(v),6))
    except AssertionError as e:
        print('❌', e)

verificar(resposta)